In [1]:
import torch
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

In [ ]:
from transformers import T5ForConditionalGeneration, T5Tokenizer

model_path = "llms"

tokenizer = T5Tokenizer.from_pretrained(model_path)
model = T5ForConditionalGeneration.from_pretrained(model_path).to(device)

c:\Users\dinhh\AppData\Local\Programs\Python\Python312\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [5]:
from tqdm import tqdm
import pandas as pd

In [8]:
splits = {'test': 'data/test-00000-of-00001.parquet'}

df_test = pd.read_parquet("hf://datasets/jhu-clsp/jfleg/" + splits["test"])

In [9]:
df_test

,sentence,corrections
0,New and new technology has been introduced to ...,[New technology has been introduced to society...
1,One possible outcome is that an environmentall...,[One possible outcome is that an environmental...
2,"Every person needs to know a bit about math , ...","[Every person needs to know a bit about math ,..."
3,While the travel company will most likely show...,[While the travel company will most likely sho...
4,Disadvantage is parking their car is very diff...,[A disadvantage is that parking their cars is ...
...,...,...
743,But I disegree this opinion because often the ...,[But I disagree with this opinion because ofte...
744,"it gives him many apprtunites in the life , an...",[It gives him many opportunities in life and I...
745,"In other words , the image in the TV comercial...","[In other words , the image in the TV commerci..."
746,Members gather money for the funeral and help ...,[Members gather money for the funeral to help ...


In [10]:
def correct_grammar(text, max_new_tokens=30):
    if not text.startswith("grammar:"):
        text = "grammar: " + text
    inputs = tokenizer(
        text,
        return_tensors="pt",
        truncation=True,
        max_length=128,
        padding="max_length"
    ).to(device)
    outputs = model.generate(
        inputs.input_ids,
        max_new_tokens=max_new_tokens,
        do_sample=False,
        num_beams=2,
        early_stopping=True
    )
    return tokenizer.decode(outputs[0], skip_special_tokens=True)

In [11]:
preds = []
for inp in tqdm(df_test["sentence"]):
    pred = correct_grammar(inp)
    preds.append(pred)

df_test["prediction"] = preds

100%|██████████| 748/748 [09:40<00:00,  1.29it/s]


In [13]:
df_test

,sentence,corrections,prediction
0,New and new technology has been introduced to ...,[New technology has been introduced to society...,New and new technology has been introduced to ...
1,One possible outcome is that an environmentall...,[One possible outcome is that an environmental...,One possible outcome is that an environmentall...
2,"Every person needs to know a bit about math , ...","[Every person needs to know a bit about math ,...","Every person needs to know a bit about math , ..."
3,While the travel company will most likely show...,[While the travel company will most likely sho...,While the travel company will most likely show...
4,Disadvantage is parking their car is very diff...,[A disadvantage is that parking their cars is ...,Disadvantage is that parking their car is very...
...,...,...,...
743,But I disegree this opinion because often the ...,[But I disagree with this opinion because ofte...,But I disegree this opinion because often the ...
744,"it gives him many apprtunites in the life , an...",[It gives him many opportunities in life and I...,"It gives him many apprtunites in the life , an..."
745,"In other words , the image in the TV comercial...","[In other words , the image in the TV commerci...","In other words , the image in the TV comercial..."
746,Members gather money for the funeral and help ...,[Members gather money for the funeral to help ...,Members gather money for the funeral and help ...


In [16]:
import sacrebleu

refs = list(zip(*df_test["corrections"]))
hyps = df_test["prediction"].tolist()

bleu = sacrebleu.corpus_bleu(hyps, refs, force=True)
print(f"Corpus BLEU: {bleu.score:.2f}")
print(bleu.format())  # Hiển thị chi tiết về n-gram precision, brevity penalty, ...


Corpus BLEU: 75.25
BLEU = 75.25 93.9/86.0/79.0/72.6 (BP = 0.912 ratio = 0.916 hyp_len = 12715 ref_len = 13886)


In [17]:
def match_any(pred, targets):
    return int(pred.strip() in [t.strip() for t in targets])

exact_matches = [
    match_any(pred, targets)
    for pred, targets in zip(df_test["prediction"], df_test["corrections"])
]

from sklearn.metrics import accuracy_score
acc = accuracy_score([1]*len(exact_matches), exact_matches)
print(f"Exact Match Accuracy: {acc:.4f}")

Exact Match Accuracy: 0.2233


In [18]:
import numpy as np
import Levenshtein

# Character-level distance
char_dists = [
    min([Levenshtein.distance(pred, tgt) for tgt in tgts])
    for pred, tgts in zip(df_test["prediction"], df_test["corrections"])
]

print(f"Avg Levenshtein char distance: {np.mean(char_dists):.2f}")


Avg Levenshtein char distance: 14.20


In [ ]:
from sklearn.metrics import f1_score

# Gán label = 1 nếu câu gốc khác với target
y_true = [int(orig.strip() != tgt[0].strip()) for orig, tgt in zip(df_test["sentence"], df_test["corrections"])]
y_pred = [int(orig.strip() != pred.strip()) for orig, pred in zip(df_test["sentence"], df_test["prediction"])]

f1 = f1_score(y_true, y_pred)
print(f"Correction F1-score: {f1:.4f}")

Correction F1-score: 0.8121
